# Lab 1 · Part 4 — Road Detection and Center Following

**Goal:** Find both road boundaries with OpenCV. Draw the road center. Use the detected center to steer the JetRacer at low speed.

In this exercise, you will:

1. Compare simple road-detection methods.
2. Detect the left and right edges of a road.
3. Calculate the center between the edges.
4. Draw the results on a live camera frame.
5. Test automatic steering with a low throttle command.

This is a **first demonstration**, not a complete autonomous-driving system. The first tests must be done with the drive wheels off the ground.

## 1. Prepare the JetRacer

1. Finish Parts 1–3. Stop the car and release the camera in those notebooks.
2. **Shut down their kernels**. Do not open two CSI camera sessions at the same time.
3. Keep the car on a stand. Keep the drive wheels clear of the table.
4. Put `jetracer_calibration.json` in the same folder as this notebook.
5. Keep a hand near the **physical car power switch**. The software STOP button cannot stop the car if the Jetson or connection freezes.
6. Do not test on the floor until the instructor checks the steering direction and the road overlay.

This notebook uses the wide CSI camera from Part 2:

```python
CSICamera(width=640, height=360,
          capture_width=1280, capture_height=720,
          capture_fps=30)
```

**Track assumption:** A dark road on a light surface. If the road is light, use the **Light road** checkbox. If road and floor have similar brightness, this method may not work.

## 2. How can OpenCV detect the road?

| Method | Main idea | Use in this exercise |
|---|---|---|
| **Thresholding** | Keep pixels lighter or darker than a chosen value | Detect the road area |
| **Contours** | Find the outline of a connected area | Find the road boundaries |
| **Canny edges** | Find sharp image changes | Compare left and right edges |
| **HSV color mask** | Select a chosen color | Possible extension for colored tape |
| **Hough lines** | Find straight line segments | Possible extension for straight tracks |

We will test **threshold + contours** and **Canny**. The first method is used for the low-speed driving test. Canny is a visual comparison. It can select floor patterns or markings instead of road boundaries.

A contour is the outline of an image region. An edge is a strong change in the image. They are not always the same thing.

## 3. Load OpenCV and the saved car calibration

The next cell uses the JSON file from Part 1. It does not change that file. No new packages are required on the prepared JetRacer image.

**Do not enable motion yet.**

In [ ]:
import json
import time
import threading
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar

CONFIG_FILE = Path('jetracer_calibration.json')
if not CONFIG_FILE.exists():
    raise FileNotFoundError('Copy jetracer_calibration.json into this folder.')

with CONFIG_FILE.open('r') as f:
    settings = json.load(f)

car = NvidiaRacecar()
car.throttle = 0.0
car.steering_gain = float(settings['steering_gain'])
car.steering_offset = float(settings['steering_offset'])
car.throttle_gain = float(settings['throttle_gain'])
car.steering = 0.0
car.steering_motor.throttle = car.steering_offset

print('Car:', settings['car_id'])
print('Steering gain:', car.steering_gain)
print('Steering offset:', car.steering_offset)
print('Throttle gain:', car.throttle_gain)
print('Motor: STOP')

## 4. Make a sample track image

Start without moving the car. The example has a dark road and a light floor.

OpenCV uses `(x, y)` coordinates. The image center is at `x = 320` in a 640-pixel-wide image.

In [ ]:
# A simple test road. This image is generated by OpenCV.
sample = np.full((360, 640, 3), 210, dtype=np.uint8)
road_corners = np.array([[275, 145], [365, 145],
                         [545, 359], [95, 359]], dtype=np.int32)
cv2.fillPoly(sample, [road_corners], (45, 45, 45))

ok, jpg = cv2.imencode('.jpg', sample)
if ok:
    display(widgets.Image(value=jpg.tobytes(), format='jpeg', width=640))

## 5. Convert the image to grayscale

A grayscale image stores brightness, not color. Values range from 0 (black) to 255 (white).

Try changing the threshold from `110` to `70`. The mask will show which pixels are selected.

In [ ]:
gray = cv2.cvtColor(sample, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (5, 5), 0)

# Select the dark road: dark pixels become white in the mask.
_, mask = cv2.threshold(gray, 110, 255, cv2.THRESH_BINARY_INV)

ok, jpg = cv2.imencode('.jpg', mask)
if ok:
    display(widgets.Image(value=jpg.tobytes(), format='jpeg', width=640))

## 6. Compare the result with Canny edges

Canny highlights rapid brightness changes. The road outline is visible as bright lines.

Canny can also find edges that are **not** road boundaries. This is why we inspect the result before driving.

In [ ]:
edges = cv2.Canny(gray, 60, 150)

ok, jpg = cv2.imencode('.jpg', edges)
if ok:
    display(widgets.Image(value=jpg.tobytes(), format='jpeg', width=640))

## 7. Find both road boundaries

We inspect **three scan lines** in the lower image, at `y = 220`, `265`, and `310`.

For each scan line:

1. Find the left road boundary.
2. Find the right road boundary.
3. Calculate the midpoint: `(left + right) / 2`.
4. Draw the three points.

We ignore results if the boundaries are too close, too far apart, or inconsistent. This is a simple check, not a guarantee that the correct road was found.

The function below contains the detection steps. The same function will be used for the live camera.

In [ ]:
def detect_road(frame, method='Threshold + contours', level=110,
                canny_level=60, light_road=False):
    height, width = frame.shape[:2]
    middle = width // 2
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    if method == 'Threshold + contours':
        kind = cv2.THRESH_BINARY if light_road else cv2.THRESH_BINARY_INV
        _, view = cv2.threshold(gray, level, 255, kind)
        view[:150, :] = 0             # Ignore the upper image.
        kernel = np.ones((5, 5), np.uint8)
        view = cv2.morphologyEx(view, cv2.MORPH_CLOSE, kernel)

        # Keep only the largest connected area.
        contours, _ = cv2.findContours(
            view, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        road_mask = np.zeros_like(view)
        if contours:
            road = max(contours, key=cv2.contourArea)
            if cv2.contourArea(road) > 8000:
                cv2.drawContours(road_mask, [road], -1, 255, -1)
        view = road_mask
    else:
        view = cv2.Canny(gray, canny_level, canny_level * 2)
        view[:150, :] = 0

    image = frame.copy()
    left_points = []
    right_points = []
    centers = []
    rows = [220, 265, 310]

    for y in rows:
        if method == 'Threshold + contours':
            xs = np.where(view[y] > 0)[0]
            if len(xs) < 2:
                continue
            left = int(xs[0])
            right = int(xs[-1])
        else:
            # Search a 5-pixel band for edge pixels on each side.
            band = np.any(view[y - 2:y + 3] > 0, axis=0)
            xs = np.where(band)[0]
            left_edges = xs[xs < middle]
            right_edges = xs[xs > middle]
            if len(left_edges) == 0 or len(right_edges) == 0:
                continue
            left = int(left_edges[-1])
            right = int(right_edges[0])

        road_width = right - left
        if road_width < 90 or road_width > 580:
            continue

        left_points.append((left, y))
        right_points.append((right, y))
        centers.append(((left + right) // 2, y))

    # Require at least 2 rows, including the nearest scan line.
    found = (len(centers) >= 2 and centers[-1][1] == 310)
    if found:
        x_values = [p[0] for p in centers]
        if max(x_values) - min(x_values) > 130:
            found = False

    # Draw every detected scan point, even if detection fails the check.
    for left, right, center in zip(left_points, right_points, centers):
        cv2.circle(image, left, 6, (0, 0, 255), -1)        # Left: red
        cv2.circle(image, right, 6, (0, 255, 0), -1)       # Right: green
        cv2.circle(image, center, 5, (0, 255, 255), -1)    # Center: yellow
        cv2.line(image, left, right, (180, 180, 180), 1)

    if len(left_points) > 1:
        cv2.polylines(image, [np.array(left_points)], False, (0, 0, 255), 3)
        cv2.polylines(image, [np.array(right_points)], False, (0, 255, 0), 3)
        cv2.polylines(image, [np.array(centers)], False, (0, 255, 255), 2)

    cv2.line(image, (middle, 150), (middle, 359), (255, 255, 0), 2)
    target_x = None

    if found:
        target_x = int(np.mean([p[0] for p in centers]))
        cv2.circle(image, (target_x, 310), 9, (255, 0, 255), -1)
        cv2.putText(image, 'ROAD FOUND', (15, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    else:
        cv2.putText(image, 'ROAD NOT RELIABLE', (15, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    return image, view, found, target_x

## 8. Test the detector on the sample image

The red marks show the left boundary. The green marks show the right boundary. Yellow marks show the road midpoint. The cyan vertical line shows the **camera center**.

Try the Canny method too. Compare the results. Do **not** use Canny for the first driving test.

In [ ]:
result, debug, found, target_x = detect_road(sample)
print('Road found:', found)
print('Road midpoint x:', target_x)

ok, jpg = cv2.imencode('.jpg', result)
if ok:
    display(widgets.Image(value=jpg.tobytes(), format='jpeg', width=640))

result_canny, _, found_canny, target_canny = detect_road(
    sample, method='Canny edges'
)
print('Canny midpoint:', target_canny)
ok, jpg = cv2.imencode('.jpg', result_canny)
if ok:
    display(widgets.Image(value=jpg.tobytes(), format='jpeg', width=640))

## 9. Convert road position into a steering command

The camera center is `x = 320`. A road center to the right has a positive error.

We use a small **P controller** (proportional controller):

`error = (road_center_x - 320) / 320`

`steering = Kp × error`

- `Kp` controls how strongly the car reacts.
- **Invert steering** changes the steering direction if needed.
- A steering limit stops the command from becoming too large.

This simple controller does not measure vehicle speed or predict sharp curves. We will study PID control later.

In [ ]:
# Example: the road center is 80 pixels to the right.
road_center_x = 400
error = (road_center_x - 320) / 320.0
Kp = 0.5
steering = Kp * error

print('Error:', round(error, 3))
print('Steering:', round(steering, 3))

## 10. Set up the live camera controls

Start with **Preview only**. Use the controls to compare the methods and adjust the threshold.

- **Start Camera / Stop Camera:** Open or release the CSI camera.
- **Method:** Compare threshold + contours with Canny.
- **Threshold:** Change the brightness limit.
- **Canny low:** Change the edge sensitivity.
- **Light road:** Select a bright road instead of a dark road.
- **Kp / Invert steering:** Set the direction and strength of automatic steering.
- **Throttle:** Set a low forward command; it does nothing until driving is enabled.
- **Enable drive:** Permit automatic control after valid detection.
- **STOP CAR:** Set throttle to zero and disarm the car.

**Canny mode is preview only.** The drive checkbox works only in threshold mode. Do not enable driving if the road overlay is incorrect.

In [ ]:
camera = None
frame_count = 0
valid_count = 0
last_frame_at = 0.0
watchdog_running = True

start_button = widgets.Button(description='Start Camera', button_style='success')
stop_button = widgets.Button(description='Stop Camera', button_style='warning', disabled=True)
stop_car_button = widgets.Button(description='STOP CAR', button_style='danger')

method_box = widgets.Dropdown(
    options=['Threshold + contours', 'Canny edges'],
    value='Threshold + contours', description='Method'
)
threshold_slider = widgets.IntSlider(
    value=110, min=30, max=220, step=5,
    description='Threshold', continuous_update=False
)
canny_slider = widgets.IntSlider(
    value=60, min=20, max=140, step=5,
    description='Canny low', continuous_update=False
)
light_road = widgets.Checkbox(value=False, description='Light road')
kp_slider = widgets.FloatSlider(
    value=0.50, min=0.10, max=1.00, step=0.05,
    description='Kp', continuous_update=False
)
invert_steering = widgets.Checkbox(value=False, description='Invert steering')
max_steering = min(0.35, float(settings['steering_max']), -float(settings['steering_min']))

max_speed = min(0.13, float(settings['normal']))
start_speed = min(float(settings['start']), max_speed)
throttle_slider = widgets.FloatSlider(
    value=start_speed, min=0.0, max=max_speed, step=0.01,
    description='Throttle', continuous_update=False
)
enable_drive = widgets.Checkbox(value=False, description='Enable drive (supervised)')

status = widgets.HTML(value='<b>Camera:</b> stopped; <b>Drive:</b> disarmed')
readout = widgets.HTML(value='No road result yet.')
preview = widgets.Image(format='jpeg', width=640, height=360)
mask_preview = widgets.Image(format='jpeg', width=320, height=180)

## 11. Process each camera frame

The callback reads the latest JetCam frame. It detects the road, draws the result, and updates the image widgets.

The notebook checks one of every three frames. This keeps the work small on the Jetson Nano. It is **not a fixed control frequency**.

If the two boundaries are not reliable, the car stops and the drive checkbox turns off. A watchdog also stops the car if camera frames stop arriving.

In [ ]:
def stop_car(button=None):
    enable_drive.value = False
    car.throttle = 0.0
    car.steering = 0.0


def drive_changed(change):
    global valid_count
    valid_count = 0
    car.throttle = 0.0
    if change['new']:
        if camera is None or not camera.running:
            enable_drive.value = False
        elif method_box.value != 'Threshold + contours':
            enable_drive.value = False
            status.value = '<b>Drive:</b> Canny mode is preview only.'


def process_frame(change):
    global frame_count, valid_count, last_frame_at
    last_frame_at = time.monotonic()
    frame_count += 1
    if frame_count % 3 != 0:
        return

    try:
        frame = change['new'].copy()
        image, mask, found, target_x = detect_road(
            frame, method_box.value, threshold_slider.value,
            canny_slider.value, light_road.value
        )

        error = 0.0
        command = 0.0
        if found:
            error = (target_x - 320) / 320.0
            command = kp_slider.value * error
            if invert_steering.value:
                command = -command
            command = max(-max_steering, min(max_steering, command))
            valid_count += 1
        else:
            valid_count = 0

        if enable_drive.value:
            if not found or method_box.value != 'Threshold + contours':
                stop_car()       # Lost road: stop and require a new arm.
                status.value = '<b>Drive:</b> stopped; check the road.'
            elif valid_count >= 3:
                car.steering = command
                car.throttle = throttle_slider.value

        label = 'FOUND' if found else 'NOT RELIABLE'
        readout.value = (
            '<b>Road:</b> {} &nbsp; <b>Target x:</b> {} &nbsp; '
            '<b>Error:</b> {:.2f} &nbsp; <b>Steering:</b> {:.2f}'
        ).format(label, target_x, error, command)

        ok, jpg = cv2.imencode('.jpg', image)
        if ok:
            preview.value = jpg.tobytes()
        ok, jpg = cv2.imencode('.jpg', mask)
        if ok:
            mask_preview.value = jpg.tobytes()
    except Exception as error:
        stop_car()
        status.value = '<b>Processing error:</b> ' + str(error)


stop_car_button.on_click(stop_car)
enable_drive.observe(drive_changed, names='value')


def watch_camera():
    while watchdog_running:
        if enable_drive.value and (time.monotonic() - last_frame_at > 0.8):
            stop_car()
            status.value = '<b>Drive:</b> stopped; camera frame timeout.'
        time.sleep(0.15)

watchdog_thread = threading.Thread(target=watch_camera)
watchdog_thread.daemon = True
watchdog_thread.start()

## 12. Start and stop the CSI camera

Keep the wheels above the table when you first start the camera.

The Start button creates a new camera session. The Stop button stops the car, stops camera capture, and releases the CSI device. Use Stop before restarting this notebook.

In [ ]:
def start_camera(button):
    global camera, frame_count, valid_count, last_frame_at
    if camera is not None:
        return

    stop_car()
    start_button.disabled = True
    status.value = '<b>Camera:</b> opening...'
    frame_count = 0
    valid_count = 0
    last_frame_at = time.monotonic()

    try:
        camera = CSICamera(
            width=640, height=360,
            capture_width=1280, capture_height=720,
            capture_fps=30
        )
        camera.observe(process_frame, names='value')
        camera.running = True
        stop_button.disabled = False
        status.value = '<b>Camera:</b> running (640 x 360); <b>Drive:</b> disarmed'
    except Exception as error:
        stop_car()
        if camera is not None:
            try:
                camera.running = False
                camera.cap.release()
            except Exception:
                pass
        camera = None
        start_button.disabled = False
        status.value = '<b>Camera error:</b> ' + str(error)


def stop_camera(button=None):
    global camera
    stop_car()
    if camera is not None:
        camera.running = False
        camera.unobserve(process_frame, names='value')
        camera.cap.release()
        camera = None

    preview.value = b''
    mask_preview.value = b''
    start_button.disabled = False
    stop_button.disabled = True
    status.value = '<b>Camera:</b> stopped and released; <b>Drive:</b> disarmed'


start_button.on_click(start_camera)
stop_button.on_click(stop_camera)

## 13. Run the live dashboard

**Exercise A — Detection only**

1. Keep **Enable drive** off.
2. Press **Start Camera**.
3. Select **Threshold + contours**.
4. Adjust **Threshold** until the road is white in the mask and the floor is black.
5. Check that the red and green lines match the real road boundaries.
6. Move the car by hand. Watch the yellow road center move.
7. Select **Canny edges**. Compare the result. Return to threshold mode.

**Exercise B — Steering test (wheels off the floor)**

1. Set **Throttle = 0.00**.
2. Select **Threshold + contours**. Confirm both boundaries are correct.
3. Select **Enable drive**. The system can now command steering, but not throttle.
4. Turn the car by hand. Check that the front wheels turn **toward** the displayed road center.
5. If steering turns the wrong way, disarm, select **Invert steering**, and test again.

**Exercise C — Supervised low-speed driving**

1. Stop the car and disarm the drive.
2. Ask the instructor to check the road overlay, steering direction, and clear floor area.
3. Put the car on the track. Set **Throttle** to `0.12` or a lower tested value.
4. Start with a straight road section. Select **Enable drive**.
5. Press **STOP CAR** if the path is incorrect. Use the physical power switch if software control does not respond.

**Note:** The camera center is only an approximation of the vehicle center. A narrow, curved, or poorly lit track can give incorrect results.

In [ ]:
camera_panel = widgets.VBox([
    widgets.HTML('<h3>Wide CSI Camera: 640 x 360</h3>'),
    widgets.HBox([start_button, stop_button, stop_car_button]),
    status,
    preview,
    widgets.HTML('<b>Road mask or Canny edges</b>'),
    mask_preview,
    readout
])

settings_panel = widgets.VBox([
    widgets.HTML('<h3>Road detection</h3>'),
    method_box, threshold_slider, canny_slider, light_road,
    widgets.HTML('<h3>Automatic steering (P controller)</h3>'),
    kp_slider, invert_steering,
    widgets.HTML('<h3>Drive</h3>'),
    throttle_slider, enable_drive
])

display(widgets.VBox([camera_panel, settings_panel]))

## 14.  questions

1. What changes when you increase the threshold value?
2. What happens when only **one** road edge is visible?
3. Why can Canny detect a line that is not part of the road?
4. What does `Kp` change? Does a larger `Kp` always help?
5. What will the car do when the road cannot be detected reliably?
6. What is the difference between `camera.running = False` and `camera.cap.release()`?

**Optional task:** Set the threshold so that the detected road remains stable under two different light levels. Note the threshold values you used.

## 15. Finish safely

1. Press **STOP CAR**.
2. Untick **Enable drive**.
3. Press **Stop Camera**.
4. Run the cleanup cell below.
5. Shut down this notebook kernel before you open another JetRacer notebook.

If the camera is stuck after you stop the notebook, close other camera applications. Then use a Jetson terminal:

```bash
sudo systemctl restart nvargus-daemon
```

Use this command only when no camera program is using the device.

In [ ]:
stop_camera()
watchdog_running = False
car.throttle = 0.0
car.steering = 0.0
print('Camera released. Car stopped. Part 4 complete.')